In [1]:
# This sample code demonstrates key concepts in RDD:
#    * lazy evaluation (delayed until action)
#    * pipelined via function composition (for narrow ops)
#    * lineage
#    * basic vs derived operations (e.g., join)
#
# But note this is simplified RDD implementation:
#    * data are not partitioned
#    * computation is not done in parallel
#    * no shuffling
#

from functools import reduce as _reduce
from itertools import chain
from collections.abc import Iterable

In [2]:
def _do_python_join(rdd, other, dispatch):
    left = rdd.map(lambda x: (x[0], ('left', x[1])))
    right = other.map(lambda x: (x[0], ('right', x[1])))
    return left.union(right).groupByKey().flatMapValues(dispatch)

def python_join(rdd, other):
    def dispatch(seq):
        ########################################
        # TODO
        # fill in missing code (20 points)
        left_vals = [v for tag, v in seq if tag == 'left']
        right_vals = [v for tag, v in seq if tag == 'right']
        for l in left_vals:
            for r in right_vals:
                yield (l, r)

    return _do_python_join(rdd, other, dispatch)

def python_left_outer_join(rdd, other):
    def dispatch(seq):
        ########################################
        # TODO
        # fill in missing code (5 points)
        left_vals = [v for tag, v in seq if tag == 'left']
        right_vals = [v for tag, v in seq if tag == 'right']
        if not right_vals:
            right_vals = [None]
        for l in left_vals:
            for r in right_vals:
                yield (l, r)

    return _do_python_join(rdd, other, dispatch)

def python_right_outer_join(rdd, other):
    def dispatch(seq):
        ########################################
        # TODO
        # fill in missing code (5 points)
        left_vals = [v for tag, v in seq if tag == 'left']
        right_vals = [v for tag, v in seq if tag == 'right']
        if not left_vals:
            left_vals = [None]
        for l in left_vals:
            for r in right_vals:
                yield (l, r)

    return _do_python_join(rdd, other, dispatch)

def python_full_outer_join(rdd, other):
    def dispatch(seq):
        ########################################
        # TODO
        # fill in missing code (5 points)
        left_vals = [v for tag, v in seq if tag == 'left']
        right_vals = [v for tag, v in seq if tag == 'right']
        if not left_vals:
            left_vals = [None]
        if not right_vals:
            right_vals = [None]
        for l in left_vals:
            for r in right_vals:
                yield (l, r)

    return _do_python_join(rdd, other, dispatch)

In [3]:
class RDD:
    def __init__(self, data=None, prev=None, func=None, label=None,
                 op=None, wide=False):
        self._data = data
        self._prev = prev
        self._func = func
        self._label = label
        self._op = op
        self._is_wide = wide

        self._cache = None
        self._cache_enabled = False

    @classmethod
    def parallelize(cls, data):
        return cls(data=data, label="parallelize")

    def cache(self):
        self._cache_enabled = True
        return self

    def unpersist(self):
        self._cache = None
        self._cache_enabled = False
        return self

    def _not_pipelinable(self):
        return self._cache_enabled or self._cache is not None or self._is_wide

    def _is_pipelinable(self):
        return not self._not_pipelinable()

    # ---------- Narrow ops (return PipelinedRDD) ----------
    # note element-level function, e.g., f, is converted into partition-level
    #   function, e.g., func,
    #   which takes an iterator (over elements in the partition)
    #   and returns another iterator (over its output)
    # e.g.,
    #    func = lambda it: map(f, it)
    #
    # these partition-based func's will be fused in PipelinedRDD

    def map(self, f):
        return PipelinedRDD(self, lambda it: map(f, it), label="map")

    def flatMap(self, f):
        return PipelinedRDD(self, lambda it: chain.from_iterable(map(f, it)), label="flatMap")

    def filter(self, f):
        return PipelinedRDD(self, lambda it: filter(f, it), label="filter")

    def mapValues(self, f):
        return PipelinedRDD(self, lambda it: ((k, f(v)) for k, v in it), label="mapValues")

    def flatMapValues(self, f):
        ########################################
        # TODO
        # fill in missing code (10 points)
        return PipelinedRDD(
            self, 
            lambda it: ((k, v_out) for k, v_in in it for v_out in f(v_in)), 
            label="flatMapValues"
        )

    # ---------- Wide ops ----------

    def union(self, other):
        return RDD(prev=(self, other), op="union", label="union", wide=True)

    def groupByKey(self):
        return RDD(prev=(self,), op="groupByKey", label="groupByKey", wide=True)

    def reduceByKey(self, func):
        return RDD(prev=(self,), func=func, op="reduceByKey", label="reduceByKey", wide=True)

    # derived operator
    def join(self, other):
        return python_join(self, other)

    def leftOuterJoin(self, other):
        return python_left_outer_join(self, other)

    def rightOuterJoin(self, other):
        return python_right_outer_join(self, other)

    def fullOuterJoin(self, other):
        return python_full_outer_join(self, other)

    # ---------- Execution ----------

    def _compute(self):
        if self._cache is not None:
            return self._cache

        # source RDD (created via parallelize())
        if self._data is not None and self._func is None and self._op is None:
            result = self._data

        elif self._op == "union":
            ########################################
            # TODO
            # fill in missing code (10 points)
            result = chain(self._prev[0]._compute(), self._prev[1]._compute())
        elif self._op == "groupByKey":
            grouped = {}
            for k, v in self._prev[0]._compute():
                grouped.setdefault(k, []).append(v)
            result = grouped.items()

        elif self._op == "reduceByKey":
            ########################################
            # TODO
            # fill in missing code (15 points)
            grouped = {}
            for k, v in self._prev[0]._compute():
                if k not in grouped:
                    grouped[k] = v
                else:
                    grouped[k] = self._func(grouped[k], v)
            result = grouped.items()

        elif self._func is not None:
            # must be pipelined rdd
            assert isinstance(self, PipelinedRDD)

            parent = self._prev._compute()
            result = self._func(iter(parent))

        else:
            raise ValueError(f"Unknown op: {self._op}")

        if self._cache_enabled:
            result = self._materialize(result)
            self._cache = result

        return result

    @staticmethod
    def _materialize(result):
        if isinstance(result, dict):
            return list(result.items())
        if isinstance(result, Iterable) and not isinstance(result, (str, bytes)):
            return list(result)
        return result

    def collect(self):
        return self._materialize(self._compute())

    def reduce(self, func):
        return _reduce(func, self._compute())

    def lineage(self, tab=0):
        name = self._label or self._op or "source"
        print(" " * tab + str(name))
        if self._prev:
            parents = self._prev if isinstance(self._prev, tuple) else (self._prev,)
            for p in parents:
                p.lineage(tab + 2)

    def __repr__(self):
        return f"{self._label}(op={self._op})"


class PipelinedRDD(RDD):
    def __init__(self, prev, transform, label=None):
        if isinstance(prev, PipelinedRDD) and prev._is_pipelinable():
            # fuse
            def composed(it):
                return transform(prev._func(it))

            super().__init__(
                prev=prev._prev,
                func=composed,
                label=label,
            )
        else:
            # new stage
            super().__init__(
                prev=prev,
                func=transform,
                label=label,
            )

In [4]:
# test all RDD operations except for join

def run_rdd_tests():
    # ---------- BASIC ----------
    assert RDD.parallelize([1, 2, 3]).collect() == [1, 2, 3]

    # ---------- MAP ----------
    assert RDD.parallelize([1, 2, 3]).map(lambda x: x * 2).collect() == [2, 4, 6]

    # ---------- FILTER ----------
    assert RDD.parallelize([1, 2, 3, 4]).filter(lambda x: x % 2 == 0).collect() == [2, 4]

    # ---------- FLATMAP ----------
    assert RDD.parallelize([1, 2]).flatMap(lambda x: [x, x + 10]).collect() == [1, 11, 2, 12]

    # ---------- MAP VALUES ----------
    assert RDD.parallelize([("a", 1), ("b", 2)]).mapValues(lambda v: v * 10).collect() == [
        ("a", 10),
        ("b", 20),
    ]

    # ---------- FLATMAP VALUES ----------
    assert RDD.parallelize([("a", [1, 2])]).flatMapValues(lambda v: v).collect() == [
        ("a", 1),
        ("a", 2),
    ]

    # ---------- UNION ----------
    left = RDD.parallelize([1, 2])
    right = RDD.parallelize([3, 4])
    assert left.union(right).collect() == [1, 2, 3, 4]

    # ---------- GROUP BY KEY ----------
    rdd = RDD.parallelize([("a", 1), ("a", 2), ("b", 3)]).groupByKey()
    result = sorted((k, sorted(v)) for k, v in rdd.collect())
    assert result == [("a", [1, 2]), ("b", [3])]

    # ---------- REDUCE BY KEY ----------
    rdd = RDD.parallelize([("a", 1), ("a", 2), ("b", 3)]).reduceByKey(lambda x, y: x + y)
    assert sorted(rdd.collect()) == [("a", 3), ("b", 3)]

    # ---------- REDUCE ----------
    assert RDD.parallelize([1, 2, 3, 4]).reduce(lambda x, y: x + y) == 10

    # ---------- CACHE ----------
    counter = {"calls": 0}

    def f(x):
        counter["calls"] += 1
        return x * 2

    rdd = RDD.parallelize([1, 2, 3]).map(f).cache()
    assert rdd.collect() == [2, 4, 6]
    assert rdd.collect() == [2, 4, 6]
    assert counter["calls"] == 3  # only once

    # ---------- UNPERSIST ----------
    rdd.unpersist()
    assert rdd.collect() == [2, 4, 6]
    assert counter["calls"] == 6  # recomputed

    # ---------- PIPELINING (FUSION) ----------
    base = RDD.parallelize([1, 2, 3])
    rdd = base.map(lambda x: x + 1).filter(lambda x: x % 2 == 0).map(lambda x: x * 10)

    assert isinstance(rdd, PipelinedRDD)
    assert rdd.collect() == [20, 40]

    # ---------- WIDE OP BREAKS PIPELINE ----------
    rdd = (
        RDD.parallelize([("a", 1), ("a", 2)])
        .map(lambda x: x)
        .groupByKey()
        .map(lambda kv: (kv[0], sorted(kv[1])))
    )

    assert rdd.collect() == [("a", [1, 2])]

    # ---------- MATERIALIZE ----------
    assert RDD._materialize({"a": 1}) == [("a", 1)]
    assert RDD._materialize((x for x in [1, 2])) == [1, 2]
    assert RDD._materialize("abc") == "abc"
    assert RDD._materialize(b"abc") == b"abc"

    # ---------- LINEAGE (SMOKE TEST) ----------
    # Just ensure it doesn't crash
    RDD.parallelize([1]).map(lambda x: x).filter(lambda x: x).lineage()

    print("All RDD tests passed ✅")


if __name__ == "__main__":
    run_rdd_tests()

filter
  parallelize
All RDD tests passed ✅


In [5]:
# test join operations

def sort_pairs(rows):
    return sorted(rows)

def run_join_tests():
    # ---------- INNER JOIN ----------
    left = RDD.parallelize([(1, "a"), (2, "b"), (3, "c")])
    right = RDD.parallelize([(1, "x"), (2, "y"), (4, "z")])
    assert sort_pairs(python_join(left, right).collect()) == [
        (1, ("a", "x")),
        (2, ("b", "y")),
    ]

    # multiple matches → cartesian product
    left = RDD.parallelize([(1, "a1"), (1, "a2")])
    right = RDD.parallelize([(1, "x1"), (1, "x2")])
    assert sort_pairs(python_join(left, right).collect()) == [
        (1, ("a1", "x1")),
        (1, ("a1", "x2")),
        (1, ("a2", "x1")),
        (1, ("a2", "x2")),
    ]

    # no overlap
    assert python_join(
        RDD.parallelize([(1, "a")]),
        RDD.parallelize([(2, "b")])
    ).collect() == []

    # ---------- LEFT OUTER JOIN ----------
    left = RDD.parallelize([(1, "a"), (2, "b"), (3, "c")])
    right = RDD.parallelize([(1, "x"), (2, "y")])
    assert sort_pairs(python_left_outer_join(left, right).collect()) == [
        (1, ("a", "x")),
        (2, ("b", "y")),
        (3, ("c", None)),
    ]

    # left only
    left = RDD.parallelize([(1, "a1"), (1, "a2")])
    right = RDD.parallelize([])
    assert sort_pairs(python_left_outer_join(left, right).collect()) == [
        (1, ("a1", None)),
        (1, ("a2", None)),
    ]

    # ---------- RIGHT OUTER JOIN ----------
    left = RDD.parallelize([(1, "a"), (2, "b")])
    right = RDD.parallelize([(1, "x"), (3, "z")])
    assert sort_pairs(python_right_outer_join(left, right).collect()) == [
        (1, ("a", "x")),
        (3, (None, "z")),
    ]

    # right only
    left = RDD.parallelize([])
    right = RDD.parallelize([(1, "x"), (2, "y")])
    assert sort_pairs(python_right_outer_join(left, right).collect()) == [
        (1, (None, "x")),
        (2, (None, "y")),
    ]

    # ---------- FULL OUTER JOIN ----------
    left = RDD.parallelize([(1, "a"), (2, "b")])
    right = RDD.parallelize([(1, "x"), (3, "z")])
    assert sort_pairs(python_full_outer_join(left, right).collect()) == [
        (1, ("a", "x")),
        (2, ("b", None)),
        (3, (None, "z")),
    ]

    # cartesian product when both sides have multiple values
    left = RDD.parallelize([(1, "a1"), (1, "a2")])
    right = RDD.parallelize([(1, "x1"), (1, "x2")])
    assert sort_pairs(python_full_outer_join(left, right).collect()) == [
        (1, ("a1", "x1")),
        (1, ("a1", "x2")),
        (1, ("a2", "x1")),
        (1, ("a2", "x2")),
    ]

    # left only
    assert python_full_outer_join(
        RDD.parallelize([(1, "a")]),
        RDD.parallelize([])
    ).collect() == [(1, ("a", None))]

    # right only
    assert python_full_outer_join(
        RDD.parallelize([]),
        RDD.parallelize([(1, "x")])
    ).collect() == [(1, (None, "x"))]

    # both empty
    assert python_full_outer_join(
        RDD.parallelize([]),
        RDD.parallelize([])
    ).collect() == []

    print("All tests passed ✅")


if __name__ == "__main__":
    run_join_tests()

All tests passed ✅


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import Row

# 1. Start a basic session
spark = SparkSession.builder.appName("HW5_Bypass").getOrCreate()

# 2. Create the 'country' DataFrame with a 'Code' column
country_data = [Row(Code="USA"), Row(Code="AFG"), Row(Code="NLD")]
country = spark.createDataFrame(country_data)

# 3. Create the 'city' DataFrame with a 'CountryCode' column
city_data = [Row(CountryCode="USA"), Row(CountryCode="AFG")]
city = spark.createDataFrame(city_data)

# 4. Create the 'cl' (countrylanguage) DataFrame with a 'CountryCode' column
cl_data = [Row(CountryCode="USA"), Row(CountryCode="NLD")]
cl = spark.createDataFrame(cl_data)

print("Success: DataFrames created in memory! No files needed.")

print("\n=== QUERY 1 PLAN (Broadcast) ===")
country.hint('broadcast').join(city, country.Code == city.CountryCode).explain()

print("\n=== QUERY 2 PLAN (Shuffle Hash) ===")
country.hint('shuffle_hash').join(city, country.Code == city.CountryCode).explain()

print("\n=== QUERY 3 PLAN (Sort Merge) ===")
cl.hint('merge').join(country, country.Code == cl.CountryCode).explain()

26/04/14 22:52:30 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


Success: DataFrames created in memory! No files needed.

=== QUERY 1 PLAN (Broadcast) ===
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- BroadcastHashJoin [Code#0], [CountryCode#1], Inner, BuildLeft, false
   :- BroadcastExchange HashedRelationBroadcastMode(List(input[0, string, false]),false), [plan_id=20]
   :  +- Filter isnotnull(Code#0)
   :     +- Scan ExistingRDD[Code#0]
   +- Filter isnotnull(CountryCode#1)
      +- Scan ExistingRDD[CountryCode#1]



=== QUERY 2 PLAN (Shuffle Hash) ===
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- ShuffledHashJoin [Code#0], [CountryCode#1], Inner, BuildLeft
   :- Exchange hashpartitioning(Code#0, 200), ENSURE_REQUIREMENTS, [plan_id=43]
   :  +- Filter isnotnull(Code#0)
   :     +- Scan ExistingRDD[Code#0]
   +- Exchange hashpartitioning(CountryCode#1, 200), ENSURE_REQUIREMENTS, [plan_id=44]
      +- Filter isnotnull(CountryCode#1)
         +- Scan ExistingRDD[CountryCode#1]



=== QUERY 3 PLAN (Sort Merge) ===
== Physica